In [0]:
%run ./00-Config

In [0]:
# COMMAND ----------
# ==========================================
# IMPORTS
# ==========================================

from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.window import Window

from pyspark.sql.types import *

from datetime import datetime

In [0]:
# COMMAND ----------
# ==========================================
# CONFIGURAÇÕES
# ==========================================

silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

gold_control_table = (
    f"{catalog_name}.control.gold_control"
)

# DIMENSIONS

dim_brewery_type_table = (
    f"{catalog_name}.gold.dim_brewery_type"
)

dim_location_table = (
    f"{catalog_name}.gold.dim_location"
)

dim_brewery_table = (
    f"{catalog_name}.gold.dim_brewery"
)

# FACT

fact_brewery_table = (
    f"{catalog_name}.gold.fact_brewery"
)


In [0]:
# COMMAND ----------
# ==========================================
# CRIA TABELA CONTROL GOLD
# ==========================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS
{gold_control_table}

(
    pipeline_name STRING,
    last_ingestion_timestamp TIMESTAMP,
    processed_at TIMESTAMP,
    records_processed BIGINT
)

USING DELTA

""")

In [0]:
# COMMAND ----------
# ==========================================
# OBTÉM ÚLTIMO TIMESTAMP
# ==========================================

control_df = spark.sql(f"""

SELECT
    MAX(last_ingestion_timestamp) AS last_ts

FROM {gold_control_table}

WHERE pipeline_name = 'gold_layer'

""")

last_ts = control_df.collect()[0]["last_ts"]

print("=" * 60)
print(f"Último timestamp GOLD: {last_ts}")
print("=" * 60)

# COMMAND ----------
# ==========================================
# LEITURA SILVER
# ==========================================

silver_df = (

    spark.read.table(silver_table)

    .filter(
        F.col("is_current") == True
    )
)

In [0]:
# COMMAND ----------
# ==========================================
# PROCESSAMENTO INCREMENTAL
# ==========================================

if last_ts:

    silver_df = (

        silver_df

        .filter(
            F.col("ingestion_timestamp") > F.lit(last_ts)
        )
    )


In [0]:
# COMMAND ----------
# ==========================================
# PADRONIZAÇÕES
# ==========================================

silver_df = (

    silver_df

    .withColumn(
        "name",
        F.trim(F.col("name"))
    )

    .withColumn(
        "brewery_type",
        F.lower(F.col("brewery_type"))
    )

    .withColumn(
        "city",
        F.trim(F.col("city"))
    )

    .withColumn(
        "state",
        F.trim(F.col("state"))
    )

    .withColumn(
        "country",
        F.upper(F.col("country"))
    )
)

In [0]:
# COMMAND ----------
# ==========================================
# REMOVE DUPLICADOS
# ==========================================

silver_df = silver_df.dropDuplicates(["id"])

In [0]:
# COMMAND ----------
# ==========================================
# DIM BREWERY TYPE
# ==========================================

dim_brewery_type_df = (

    silver_df

    .select("brewery_type")

    .filter(
        F.col("brewery_type").isNotNull()
    )

    .dropDuplicates()

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )
)

In [0]:

# COMMAND ----------
# ==========================================
# MERGE DIM BREWERY TYPE
# ==========================================

if not spark.catalog.tableExists(dim_brewery_type_table):

    (
        dim_brewery_type_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_brewery_type_table)
    )

else:

    delta_dim_type = DeltaTable.forName(
        spark,
        dim_brewery_type_table
    )

    (
        delta_dim_type.alias("target")

        .merge(

            dim_brewery_type_df.alias("source"),

            """
            target.brewery_type_sk =
            source.brewery_type_sk
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

# COMMAND ----------
# ==========================================
# DIM LOCATION
# ==========================================

dim_location_df = (

    silver_df

    .select(
        "city",
        "state",
        "country"
    )

    .filter(
        F.col("country").isNotNull()
    )

    .dropDuplicates()

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )
)

In [0]:

# COMMAND ----------
# ==========================================
# MERGE DIM LOCATION
# ==========================================

if not spark.catalog.tableExists(dim_location_table):

    (
        dim_location_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_location_table)
    )

else:

    delta_dim_location = DeltaTable.forName(
        spark,
        dim_location_table
    )

    (
        delta_dim_location.alias("target")

        .merge(

            dim_location_df.alias("source"),

            """
            target.location_sk =
            source.location_sk
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

In [0]:
# COMMAND ----------
# ==========================================
# DIM BREWERY
# ==========================================

dim_brewery_df = (

    silver_df

    .withColumn(

        "brewery_sk",

        F.sha2(
            F.col("id"),
            256
        )
    )

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )

    .select(

        "brewery_sk",

        F.col("id").alias("brewery_id"),

        F.col("name").alias("brewery_name"),

        "brewery_type_sk",

        "location_sk",

        "website_url",

        "phone",

        "ingestion_timestamp"
    )
)


In [0]:
# COMMAND ----------
# ==========================================
# MERGE DIM BREWERY
# ==========================================

if not spark.catalog.tableExists(dim_brewery_table):

    (
        dim_brewery_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_brewery_table)
    )

else:

    delta_dim_brewery = DeltaTable.forName(
        spark,
        dim_brewery_table
    )

    (
        delta_dim_brewery.alias("target")

        .merge(

            dim_brewery_df.alias("source"),

            """
            target.brewery_sk =
            source.brewery_sk
            """
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )


In [0]:
# COMMAND ----------
# ==========================================
# FACT BREWERY
# ==========================================

fact_brewery_df = (

    silver_df

    .withColumn(

        "snapshot_date",

        F.current_date()
    )

    .withColumn(

        "brewery_sk",

        F.sha2(
            F.col("id"),
            256
        )
    )

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )

    .withColumn(
        "brewery_count",
        F.lit(1)
    )

    .select(

        "snapshot_date",

        "brewery_sk",

        "brewery_type_sk",

        "location_sk",

        "brewery_count",

        "ingestion_timestamp"
    )

    .dropDuplicates([
        "snapshot_date",
        "brewery_sk"
    ])
)

In [0]:
# COMMAND ----------
# ==========================================
# VALIDAÇÕES FACT
# ==========================================

fact_brewery_df = (

    fact_brewery_df

    .filter(
        F.col("brewery_count") >= 0
    )
)

In [0]:
# COMMAND ----------
# ==========================================
# ESCRITA FACT
# ==========================================

if not spark.catalog.tableExists(fact_brewery_table):

    (
        fact_brewery_df.write

        .format("delta")

        .partitionBy("snapshot_date")

        .mode("overwrite")

        .saveAsTable(fact_brewery_table)
    )

else:

    (
        fact_brewery_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(fact_brewery_table)
    )

# COMMAND ----------
# ==========================================
# CONTROLE INCREMENTAL
# ==========================================

records_processed = fact_brewery_df.count()

max_ts = (

    fact_brewery_df

    .agg(
        F.max("ingestion_timestamp").alias("max_ts")
    )

    .collect()[0]["max_ts"]
)


In [0]:
# COMMAND ----------
# ==========================================
# SALVA CONTROLE
# ==========================================

if max_ts:

    control_data = [

        (
            "gold_layer",
            max_ts,
            datetime.utcnow(),
            records_processed
        )
    ]

    control_schema = StructType([

        StructField(
            "pipeline_name",
            StringType(),
            True
        ),

        StructField(
            "last_ingestion_timestamp",
            TimestampType(),
            True
        ),

        StructField(
            "processed_at",
            TimestampType(),
            True
        ),

        StructField(
            "records_processed",
            LongType(),
            True
        )
    ])

    control_insert_df = spark.createDataFrame(
        control_data,
        control_schema
    )

    (
        control_insert_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(gold_control_table)
    )


In [0]:

# COMMAND ----------
# ==========================================
# FINALIZAÇÃO
# ==========================================

print("=" * 60)

print("GOLD PROCESSADA COM SUCESSO")

print(f"Registros processados: {records_processed}")

print(f"Último timestamp: {max_ts}")

print("=" * 60)

In [0]:
%sql
select * from brewing.gold.dim_brewery limit 100